In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
n_samples = 1_000
treatment_effect = 0.2
p_conversion_untreated = 0.3
p_conversion_treated = p_conversion_untreated + treatment_effect

p_treated = 0.5
n_treated = int(n_samples * p_treated)
n_control = n_samples - n_treated
print(f"Treated: {n_treated}, Control: {n_control}")

user_id = np.arange(n_samples)
treated = [1] * n_treated + [0] * n_control
data = pd.DataFrame({
    "user_id": user_id,
    "treated": treated
})
def convert(treatment_status, p, delta):
    if treatment_status == 1:
        return np.random.binomial(n=1, p=p+delta)
    else:
        return np.random.binomial(n=1, p=p)
data['converted'] = data['treated'].apply(lambda x: convert(x,p_conversion_untreated, treatment_effect))
data        

Treated: 500, Control: 500


,user_id,treated,converted
0,0,1,1
1,1,1,0
2,2,1,1
3,3,1,1
4,4,1,0
...,...,...,...
995,995,0,0
996,996,0,0
997,997,0,0
998,998,0,0


In [9]:
summary = (
    data
    .groupby("treated")
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
).reset_index()

summary


,treated,users,conversions,conversion_rate
0,0,500,160,0.320
1,1,500,254,0.508


In [ ]:
r_control, r_treated = summary['conversion_rate'].values
ate = r_treated - r_control
print(f"ATE: {ate:.4f}")
import numpy as np
from scipy.stats import norm

# TODO: Fix this, this is dumb given that we have proportions already
def test_ate_zero_from_rates(p_t, n_t, p_c, n_c, *, alternative="two-sided"):
    """
    Two-proportion z-test for H0: p_t - p_c = 0, using pooled SE under H0.
    Inputs are conversion rates and sample sizes.
    """
    # If rates are exact X/n this will be integer; if rates are rounded, rounding is the best you can do.
    x_t = int(round(p_t * n_t))
    x_c = int(round(p_c * n_c))

    p_t_hat = x_t / n_t
    p_c_hat = x_c / n_c
    ate_hat = p_t_hat - p_c_hat

    # pooled rate under H0
    p_pool = (x_t + x_c) / (n_t + n_c)

    se_pooled = np.sqrt(p_pool * (1 - p_pool) * (1/n_t + 1/n_c))
    z = ate_hat / se_pooled

    if alternative == "two-sided":
        pval = 2 * (1 - norm.cdf(abs(z)))
    elif alternative == "larger":   # H1: p_t > p_c
        pval = 1 - norm.cdf(z)
    elif alternative == "smaller":  # H1: p_t < p_c
        pval = norm.cdf(z)
    else:
        raise ValueError("alternative must be 'two-sided', 'larger', or 'smaller'")

    return {
        "x_t": x_t, "x_c": x_c,
        "p_t_hat": p_t_hat, "p_c_hat": p_c_hat,
        "ate_hat": ate_hat,
        "se_pooled": se_pooled,
        "z": z,
        "p_value": pval
    }

# Example (your earlier numbers)
res = test_ate_zero_from_rates(p_t=r_treated, n_t=500, p_c=r_control, n_c=500)
print(res)

def ate_ci_from_rates(p_t, n_t, p_c, n_c, zcrit=1.96):
    ate = p_t - p_c
    se = np.sqrt(p_t*(1-p_t)/n_t + p_c*(1-p_c)/n_c)
    return ate, se, (ate - zcrit*se, ate + zcrit*se)

ate, se, ci = ate_ci_from_rates(r_treated, 500, r_control, 500)
print("ATE:", ate, "SE:", se, "95% CI:", ci)


ATE: 0.1880
{'x_t': 254, 'x_c': 160, 'p_t_hat': 0.508, 'p_c_hat': 0.32, 'ate_hat': 0.188, 'se_pooled': 0.031151500766415735, 'z': 6.0350222420963355, 'p_value': 1.5894079385958548e-09}
ATE: 0.188 SE: 0.030578947006069387 95% CI: (0.12806526386810402, 0.24793473613189598)


In [22]:
from scipy.stats import chisquare, ttest_ind

In [25]:
delta_hat = summary['conversion_rate'].values[1] - summary['conversion_rate'].values[0]
print(f"Estimated treatment effect {delta_hat}")
chi2_test = chisquare(summary['conversion_rate'].values)

t = ttest_ind(data[data['treated'] == 1]['converted'], data[data['treated'] == 0]['converted'])
# chi2_test.pvalue
t.pvalue

Estimated treatment effect 0.188


1.1760800646782223e-09